google colab setup

In [1]:
# Purpose:
# - Install dependencies
# - Ensure reproducibility
# - Prepare NLP + ML environment for pipeline

# Design principle:
# Lightweight NLP stack (no heavy LLM fine-tuning required)


!pip install -q spacy nltk textstat scikit-learn pandas numpy matplotlib seaborn shap xgboost streamlit

# Download spaCy model (linguistic feature extraction backbone)
import spacy
!python -m spacy download en_core_web_sm

# NLTK resources (used for sentiment + lexical features)
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [2]:
# imports
import numpy as np
import pandas as pd

import spacy
from nltk.sentiment import SentimentIntensityAnalyzer
import textstat

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

import shap
import matplotlib.pyplot as plt

load NLP pipeline

In [3]:
# Load spaCy model for linguistic structure analysis
nlp = spacy.load("en_core_web_sm")

# Sentiment analyzer (rule-based, no training required)
sentiment_analyzer = SentimentIntensityAnalyzer()

We transform raw text into a clean, analyzable linguistic object while preserving signals needed for:


*   psycholinguistic traits (Dark Tetrad proxies)
*   cognitive load indicators
*   temporal degradation patterns

Key principle:
Do not over-clean text, it might destroy behavioral signals.

input format design

In [4]:
# Each sample represents a single "behavioral unit"
# (e.g., Reddit post, journal entry, message, email)

sample = {
    "user_id": "U001",
    "timestamp": "2026-05-30",
    "text": "I feel like people never really listen to me anymore..."
}

cleaning pipeline (avoid aggressive preprocessing)

In [5]:
import re

def clean_text(text):
    """
    Clean text while preserving psychological and linguistic signals.

    Design rationale:
    - Excessive normalization removes emotional markers
    - Capitalization, punctuation, and repetition carry behavioral signals
    - This follows psycholinguistic preprocessing norms used in LIWC-style research
    """

    # Keep original case for emotional intensity signals
    text = text.strip()

    # Remove URLs only (non-behavioral noise)
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text

sentence segmentation

important for fatigue modelling

Fatigue is often detected through structural degradation across sentences.

In [6]:
def segment_sentences(text):
    """
    Splits text into sentences using spaCy.

    - Sentence-level variation is a strong proxy for cognitive load
    - Used in readability research and discourse analysis studies
    """

    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    return sentences

end-to-end transformation function

In [7]:
def process_sample(sample):
    """
    Converts raw input into structured linguistic object.

    Output is designed for:
    - personality feature extraction
    - fatigue tracking
    - volatility computation
    """

    text = clean_text(sample["text"])
    sentences = segment_sentences(text)

    processed = {
        "user_id": sample["user_id"],
        "timestamp": sample["timestamp"],
        "raw_text": sample["text"],
        "clean_text": text,
        "sentences": sentences,
        "num_sentences": len(sentences),
        "num_chars": len(text)
    }

    return processed

test run

In [8]:
test_sample = {
    "user_id": "U001",
    "timestamp": "2026-05-30",
    "text": "I feel like people never really listen to me anymore... It's exhausting."
}

processed = process_sample(test_sample)
processed

{'user_id': 'U001',
 'timestamp': '2026-05-30',
 'raw_text': "I feel like people never really listen to me anymore... It's exhausting.",
 'clean_text': "I feel like people never really listen to me anymore... It's exhausting.",
 'sentences': ['I feel like people never really listen to me anymore...',
  "It's exhausting."],
 'num_sentences': 2,
 'num_chars': 72}

`Personality Feature Engine (Linguistic Fingerprinting Layer)`

This is the **first real “psychological modeling” layer** of the system.

We are building:

> A probabilistic linguistic profile based on behavioral proxies used in computational psycholinguistics.

* What This Layer Does: It converts cleaned text into a **feature vector**:

```text
[Self-focus, Dominance, Emotionality, Aggression, Social Orientation, Complexity]
```

These are **not personality labels**, but measurable linguistic signals.

* Feature Design: We rely on established computational linguistics findings:

Key inspirations:

* Pennebaker et al. (LIWC framework) → self-reference & emotion markers
1. Tausczik & Pennebaker (2010) → language reflects psychological state
2. Mairesse et al. (2007) → personality prediction from text
3. Sap et al. (2014–2017) → language correlates of social traits

* Feature Categories

A. **Self-Focus** (egocentric language)

```text
I, me, my, mine
```

High values often correlate with self-referential cognition.

B. **Social Orientation**

```text
we, us, our, people, everyone
```

Represents outward vs inward focus.

C. **Emotional Intensity**

Using sentiment polarity + absolute intensity.

D. **Dominance Language**

Modal + assertive words:

```text
must, should, always, never, obviously
```
E. **Aggression Proxy**

Negative + confrontational markers.

F. **Lexical Complexity** (baseline cognitive indicator)

We will reuse:

* word diversity
* sentence structure simplicity


Without this layer, BVI is just:

> sentiment analysis + fatigue scoring

With this layer, it becomes:

> behavioral modeling under psychological proxy traits





In [9]:
from collections import Counter
import numpy as np

# Lexicons (simple but effective baseline version)

SELF_FOCUS = {"i", "me", "my", "mine"}
SOCIAL = {"we", "us", "our", "people", "everyone"}
DOMINANCE = {"must", "should", "always", "never", "obviously", "clearly"}
AGGRESSION = {"hate", "stupid", "idiot", "angry", "furious", "annoyed"}

# Feature extraction function

def extract_personality_features(processed_sample):
    """
    Converts processed text into behavioral feature vector.

    Scientific basis:
    - LIWC-style word category counting
    - psycholinguistic trait inference (non-clinical)
    """

    text = processed_sample["clean_text"].lower()
    words = text.split()

    word_counts = Counter(words)
    total_words = len(words) if len(words) > 0 else 1

    # Self-focus score
    self_focus = sum(word_counts[w] for w in SELF_FOCUS) / total_words

    # Social orientation
    social_focus = sum(word_counts[w] for w in SOCIAL) / total_words

    # Dominance language
    dominance = sum(word_counts[w] for w in DOMINANCE) / total_words

    # Aggression proxy
    aggression = sum(word_counts[w] for w in AGGRESSION) / total_words

    # Emotional intensity (simple proxy)
    sentiment_score = sentiment_analyzer.polarity_scores(processed_sample["clean_text"])["compound"]
    emotional_intensity = abs(sentiment_score)

    # Lexical diversity (TTR approximation)
    lexical_diversity = len(set(words)) / total_words

    return {
        "self_focus": self_focus,
        "social_focus": social_focus,
        "dominance": dominance,
        "aggression": aggression,
        "emotional_intensity": emotional_intensity,
        "lexical_diversity": lexical_diversity
    }

In [10]:
features = extract_personality_features(processed)
features

{'self_focus': 0.16666666666666666,
 'social_focus': 0.08333333333333333,
 'dominance': 0.08333333333333333,
 'aggression': 0.0,
 'emotional_intensity': 0.0,
 'lexical_diversity': 1.0}

output is consistent with a very small sample size (2 sentences / ~12 words), so a couple of values (like lexical diversity = 1.0) are expected artifacts rather than real signals.

Now it comes to Cognitive Fatigue engine
> How linguistic structure breaks down under cognitive strain

It is a computational proxy for cognitive load, derived from:

* linguistic simplification
* repetition tendency
* reduced lexical richness
* reduced syntactic complexity
* emotional compression (flattening or spikes)

Fatigue feature set consists of

* Sentence Compression: Shorter sentences under load.

* Lexical Reduction: Lower diversity over time.

* Repetition Increase: More repeated words or structures.

* Complexity Drop: Simpler language structures.



In [11]:
def compute_fatigue_features(processed_sample, personality_features):
    """
    Estimates cognitive fatigue using linguistic degradation signals.

    Scientific basis:
    - Cognitive Load Theory (Sweller, 1988)
    - Discourse simplification under mental effort (Graesser et al.)
    - Linguistic reduction under stress (Pennebaker & Tausczik, 2010)
    """

    sentences = processed_sample["sentences"]
    text = processed_sample["clean_text"].lower()
    words = text.split()

    total_words = len(words) if len(words) > 0 else 1

    # Sentence length (compression signal)
    sentence_lengths = [len(s.split()) for s in sentences]
    avg_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0

    # Lexical richness (already computed but re-evaluated dynamically)
    lexical_diversity = len(set(words)) / total_words

    # Repetition score
    word_counts = Counter(words)
    repetition_score = sum([count - 1 for count in word_counts.values() if count > 1]) / total_words

    # Structural simplicity proxy
    # (short sentences + low diversity = higher fatigue)
    simplicity_score = 1 / (1 + avg_sentence_length)

    # Combine into fatigue index (weighted heuristic baseline)
    fatigue_score = (
        0.35 * (1 - lexical_diversity) +
        0.30 * repetition_score +
        0.20 * simplicity_score +
        0.15 * (1 - avg_sentence_length / 20)
    )

    fatigue_score = max(0, min(1, fatigue_score))  # normalize

    return {
        "avg_sentence_length": avg_sentence_length,
        "lexical_diversity": lexical_diversity,
        "repetition_score": repetition_score,
        "simplicity_score": simplicity_score,
        "fatigue_score": fatigue_score
    }

In [12]:
fatigue = compute_fatigue_features(processed, features)
fatigue

{'avg_sentence_length': np.float64(6.0),
 'lexical_diversity': 1.0,
 'repetition_score': 0.0,
 'simplicity_score': np.float64(0.14285714285714285),
 'fatigue_score': np.float64(0.13357142857142856)}

Now we can move on to Behavioral Volatility Engine (BVI Core)

Everything so far feeds into this:

* Personality features
* Fatigue features

Now we compute:

> How much behavior *changes or destabilizes* when cognitive load interacts with linguistic traits.

We define BVI as:

> A weighted interaction between baseline behavioral tendencies and cognitive strain.

### Key idea:

Not just fatigue alone — but:

> “How does fatigue amplify or distort personality expression?”



We use a structured but interpretable model:

$$
BVI = \alpha F + \beta P_{instability} + \gamma (F \times P_{instability})
$$

Where:

* **F** = fatigue_score (from Step 4)
* **P_instability** = derived personality volatility index
* **F × P_instability** = interaction effect (core novelty)




Building Personality Instability Index

We first convert features into a single composite:

In [13]:
def compute_personality_instability(personality_features):
    """
    Converts personality proxies into a single instability score.

    Interpretation:
    - Higher dominance + aggression + self-focus
    - Lower social focus + lexical diversity
    => higher instability potential
    """

    instability = (
        0.25 * personality_features["self_focus"] +
        0.20 * personality_features["dominance"] +
        0.25 * personality_features["aggression"] +
        0.15 * personality_features["emotional_intensity"] +
        0.15 * (1 - personality_features["social_focus"])
    )

    return max(0, min(1, instability))

full BVI model

In [14]:
def compute_bvi(personality_features, fatigue_features):
    """
    Behavioral Volatility Index (BVI)

    Scientific intent:
    Models interaction between:
    - stable linguistic traits
    - cognitive degradation signals

    Inspired by:
    - interaction models in psychometrics
    - stress × personality moderation literature
    """

    # Step 1: extract components
    F = fatigue_features["fatigue_score"]
    P = compute_personality_instability(personality_features)

    # Step 2: interaction term
    interaction = F * P

    # Step 3: weighted aggregation
    BVI = (
        0.40 * F +
        0.35 * P +
        0.25 * interaction
    )

    # Normalize to 0–100 scale
    BVI_scaled = BVI * 100

    return {
        "fatigue": F,
        "personality_instability": P,
        "interaction": interaction,
        "BVI": BVI_scaled
    }

In [15]:
bvi_result = compute_bvi(features, fatigue)
bvi_result

{'fatigue': np.float64(0.13357142857142856),
 'personality_instability': 0.1958333333333333,
 'interaction': np.float64(0.026157738095238088),
 'BVI': np.float64(12.850967261904762)}

Explainability Layer (Why the model produced this BVI)

Now we move from **prediction → interpretation**.

We want to answer:

> “Which linguistic factors contributed most to the Behavioral Volatility Index?”

This is essential because:

* raw ML scores are not interpretable
* psychological modeling requires explanation, not just prediction
* GitHub reviewers value transparency + reasoning

We are opting for (Two-Level Explainability)

We will implement:

* Level 1 — Rule-based contribution breakdown (simple + reliable)

* Level 2 — SHAP-based model explanation (advanced, optional)

We start with Level 1

Why This Step Matters,

This aligns with:

* Lipton (2016) — “The Mythos of Model Interpretability”
* Doshi-Velez & Kim (2017) — interpretable ML in sensitive domains
* Psycholinguistics requirement: explanations must map to observable behavior

In oour case:

> We are explicitly modeling behavior, so interpretability is not optional — it is part of the system definition.



At this point the pipeline is:

✔ Raw text ingestion

✔ Linguistic preprocessing

✔ Personality feature extraction

✔ Cognitive fatigue modeling

✔ Interaction-based BVI score

✔ Explainability layer



contribution breakdown: decompose BVI into interpretable components

In [16]:
def explain_bvi(personality_features, fatigue_features):
    """
    Interpretable explanation layer for BVI.

    This replaces black-box interpretation with:
    - weighted contribution tracing
    - feature-level attribution

    Inspired by:
    - psychometric interpretability principles
    - additive feature attribution models
    """

    F = fatigue_features["fatigue_score"]

    P = (
        0.25 * personality_features["self_focus"] +
        0.20 * personality_features["dominance"] +
        0.25 * personality_features["aggression"] +
        0.15 * personality_features["emotional_intensity"] +
        0.15 * (1 - personality_features["social_focus"])
    )

    interaction = F * P

    contributions = {
        "fatigue_contribution": 0.40 * F,
        "personality_contribution": 0.35 * P,
        "interaction_contribution": 0.25 * interaction
    }

    total = sum(contributions.values())

    # normalize to percentages
    explanation = {k: (v / total) * 100 for k, v in contributions.items()}

    return explanation

In [17]:
explanation = explain_bvi(features, fatigue)
explanation

{'fatigue_contribution': np.float64(41.57552528124042),
 'personality_contribution': np.float64(53.335803655691095),
 'interaction_contribution': np.float64(5.088671063068487)}